# MNIST Handwritten Digit Classification with CNN

## Objectives
Classify 28×28 grayscale digits (0–9) using a convolutional neural network.

## CNN Theory (Simple English)
**Convolution** slides small filters (kernels) over the image to detect edges, textures, and shapes.  
**Pooling** downsamples spatial size.  
**Stack:** Conv → ReLU → Pool → ... → Flatten → Dense → Softmax (multi-class).

**Formula (2D conv output size):**  
$O = \lfloor (I - K + 2P) / S \rfloor + 1$ where $I$=input size, $K$=kernel, $P$=padding, $S$=stride.


In [ ]:
# Optional: install dependencies (uncomment if needed)
# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow requests yfinance

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)


## Problem Definition
Image classification assigns a label to each image. Used in OCR, quality control, medical screening.


In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]
input_shape = (28, 28, 1)
num_classes = 10

# Use part of train as validation
val_size = 10000
train_ds = tf.data.Dataset.from_tensor_slices((x_train[val_size:], y_train[val_size:]))
val_ds = tf.data.Dataset.from_tensor_slices((x_train[:val_size], y_train[:val_size]))
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))


In [ ]:
# EDA — sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, (img, label) in zip(axes.ravel(), train_ds.take(10)):
    arr = img.numpy()
    if arr.ndim == 3 and arr.shape[-1] == 1:
        arr = arr.squeeze(-1)
    ax.imshow(arr, cmap="gray" if arr.ndim == 2 else None)
    ax.set_title(str(int(label.numpy() if hasattr(label, 'numpy') else label)))
    ax.axis("off")
plt.suptitle("Sample training images")
plt.show()


In [ ]:
# Normalize and batch
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_batched = train_ds.map(preprocess).shuffle(10000).batch(64).prefetch(AUTOTUNE)
val_batched = val_ds.map(preprocess).batch(64).prefetch(AUTOTUNE)
test_batched = test_ds.map(preprocess).batch(64).prefetch(AUTOTUNE)


In [ ]:
model = models.Sequential([
    layers.Input(shape=input_shape),
    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax"),
])
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


In [ ]:
cnn_cb = [
    callbacks.ModelCheckpoint("cnn_best.keras", save_best_only=True, monitor="val_accuracy", mode="max"),
    callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor="val_accuracy", mode="max"),
]
history = model.fit(train_batched, validation_data=val_batched, epochs=25, callbacks=cnn_cb, verbose=1)


In [ ]:
test_loss, test_acc = model.evaluate(test_batched, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

# Plot history
pd.DataFrame(history.history)[["accuracy", "val_accuracy"]].plot(figsize=(8,4))
plt.title("CNN training accuracy")
plt.show()

y_true, y_pred = [], []
for images, labels in test_batched:
    preds = model.predict(images, verbose=0).argmax(axis=1)
    y_pred.extend(preds)
    y_true.extend(labels.numpy())
print(classification_report(y_true, y_pred))


In [ ]:
# Inference on one batch
for images, labels in test_batched.take(1):
    probs = model.predict(images[:3], verbose=0)
    for i in range(3):
        print("True:", int(labels[i]), "Pred:", probs[i].argmax(), "Confidence:", probs[i].max():.3f)
model.save("cnn_deployed.keras")


## Deployment Notes

1. **Serving**: Export with `model.export("saved_model")` for TensorFlow Serving, or wrap `predict` in FastAPI/Flask.
2. **Preprocessing**: Always apply the **same** scaler/encoder fitted on training data (`scaler.pkl`).
3. **Monitoring**: Track input drift, latency, and prediction distribution on live traffic.
4. **Retraining**: Schedule periodic retrain when performance drops below SLA.
5. **Security**: Do not log PII; use HTTPS and auth on inference endpoints.
